# MINA Voice API para Google Colab\n\nEste notebook levanta una API pequeña para generar voz de MINA usando la GPU de Colab.\n\nPasos rápidos:\n1. En Colab ve a **Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU**.\n2. Ejecuta las celdas en orden.\n3. Sube tu archivo `user_reference.wav` cuando lo pida.\n4. Copia la URL `https://...trycloudflare.com` que salga al final.\n5. En el `.env` de MINA pon `VOICE_REMOTE_URL=esa_url` y reinicia el bot.\n

In [1]:
# 1) Instalar dependencias\n!pip -q install chatterbox-tts fastapi uvicorn nest-asyncio python-multipart\n!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\n!chmod +x /content/cloudflared\n

In [2]:
# 2) Descargar referencia de voz desde tu PC por túnel temporal
import urllib.request
from pathlib import Path

url = 'https://interpreted-privacy-suspected-mercury.trycloudflare.com/assets/voice/user_reference.wav'
reference_path = Path('/content/user_reference.wav')
urllib.request.urlretrieve(url, reference_path)
print('Referencia lista en:', reference_path, 'bytes:', reference_path.stat().st_size)


Referencia lista en: /content/user_reference.wav bytes: 5325974


In [1]:
# MINA Voice API - reactivar Colab y crear URL nueva
import os, sys, subprocess, urllib.request, tempfile, threading, time, re
from pathlib import Path

cloudflared_path = Path('/content/cloudflared')
if not cloudflared_path.exists():
    print('Descargando cloudflared...')
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared_path)
    os.chmod(cloudflared_path, 0o755)

reference_path = Path('/content/user_reference.wav')
if not reference_path.exists():
    print('Descargando referencia de voz...')
    urllib.request.urlretrieve('https://interpreted-privacy-suspected-mercury.trycloudflare.com/assets/voice/user_reference.wav', reference_path)
print('Referencia lista:', reference_path, 'bytes:', reference_path.stat().st_size)

try:
    model
    app
    print('Modelo/API ya estaban cargados:', model_name)
except NameError:
    print('Cargando modelo/API...')
    import numpy
    print('NumPy:', numpy.__version__)
    import torch
    import torchaudio as ta
    import nest_asyncio
    import uvicorn
    from fastapi import FastAPI, Header, HTTPException
    from fastapi.responses import FileResponse
    from pydantic import BaseModel

    try:
        import perth
        if getattr(perth, 'PerthImplicitWatermarker', None) is None:
            class DummyWatermarker:
                def apply_watermark(self, wav, sample_rate=None):
                    return wav
            perth.PerthImplicitWatermarker = DummyWatermarker
    except Exception:
        pass

    try:
        from chatterbox.mtl_tts import ChatterboxMultilingualTTS
    except Exception as e:
        print('Multilingual no disponible:', repr(e))
        ChatterboxMultilingualTTS = None
    from chatterbox.tts import ChatterboxTTS

    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    USE_MULTILINGUAL = ChatterboxMultilingualTTS is not None
    print('GPU/CPU:', DEVICE)

    if USE_MULTILINGUAL:
        model = ChatterboxMultilingualTTS.from_pretrained(device=DEVICE)
        model_name = 'ChatterboxMultilingualTTS'
    else:
        model = ChatterboxTTS.from_pretrained(device=DEVICE)
        model.prepare_conditionals(str(reference_path), exaggeration=0.75)
        model_name = 'ChatterboxTTS'

    print('Modelo listo:', model_name, 'sample_rate:', model.sr)

    VOICE_TOKEN = os.environ.get('MINA_VOICE_TOKEN', '').strip()
    app = FastAPI(title='MINA Voice API')

    class GenerateRequest(BaseModel):
        text: str

    def check_auth(authorization: str | None):
        if not VOICE_TOKEN:
            return
        if authorization != f'Bearer {VOICE_TOKEN}':
            raise HTTPException(status_code=401, detail='Token inválido')

    @app.get('/health')
    def health(authorization: str | None = Header(default=None)):
        check_auth(authorization)
        return {'ok': True, 'device': DEVICE, 'model': model_name, 'sample_rate': model.sr}

    @app.post('/generate')
    def generate(req: GenerateRequest, authorization: str | None = Header(default=None)):
        check_auth(authorization)
        text = req.text.strip()[:260]
        if not text:
            raise HTTPException(status_code=400, detail='Texto vacío')
        out_path = Path(tempfile.gettempdir()) / f'mina_voice_{abs(hash(text))}_{os.getpid()}.wav'
        if USE_MULTILINGUAL:
            wav = model.generate(text, 'es', audio_prompt_path=str(reference_path), repetition_penalty=1.18, min_p=0.02, top_p=0.92, exaggeration=0.95, cfg_weight=0.60, temperature=0.96)
        else:
            wav = model.generate(text, repetition_penalty=1.12, min_p=0.03, top_p=0.94, exaggeration=0.88, cfg_weight=0.48, temperature=0.90)
        ta.save(str(out_path), wav, model.sr)
        return FileResponse(str(out_path), media_type='audio/wav', filename='mina.wav')

    nest_asyncio.apply()
    import uvicorn
    server = uvicorn.Server(uvicorn.Config(app, host='127.0.0.1', port=7860, log_level='warning'))
    threading.Thread(target=server.run, daemon=True).start()
    print('API local lista en http://127.0.0.1:7860')

proc = subprocess.Popen([str(cloudflared_path), 'tunnel', '--url', 'http://127.0.0.1:7860'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
logs = []
for _ in range(80):
    line = proc.stdout.readline()
    if line:
        logs.append(line.rstrip())
        print(line, end='')
        match = re.search(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            break
    time.sleep(1)
if not public_url:
    print('\n'.join(logs[-30:]))
    raise RuntimeError('No se pudo obtener la URL pública de cloudflared.')
print('MINA_REMOTE_READY=' + public_url)
print('VOICE_REMOTE_URL=' + public_url)


Referencia lista: /content/user_reference.wav bytes: 5325974
Cargando modelo/API...
NumPy: 1.26.4
GPU/CPU: cuda


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)


Cangjie5_TC.json:   0%|          | 0.00/1.92M [00:00<?, ?B/s]

Downloading: "https://github.com/explosion/spacy-pkuseg/releases/download/v0.0.26/spacy_ontonotes.zip" to /root/.pkuseg/spacy_ontonotes.zip
100%|██████████| 34567143/34567143 [00:00<00:00, 147812010.86it/s]


loaded PerthNet (Implicit) at step 250,000
Modelo listo: ChatterboxMultilingualTTS sample_rate: 24000
API local lista en http://127.0.0.1:7860
2026-07-12T15:04:47Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-07-12T15:04:47Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-07-12T15:04:51Z INF +--------------------------------------------------------------------------------------------+
2026-07-12T15:04:51Z INF |  Your quick Tunnel has been cre

In [4]:
# 4) Abrir túnel público con Cloudflare\nimport re\nimport subprocess\nimport threading\nimport time\n\nproc = subprocess.Popen(\n    ['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'],\n    stdout=subprocess.PIPE,\n    stderr=subprocess.STDOUT,\n    text=True,\n)\n\npublic_url = None\n\ndef read_logs():\n    global public_url\n    for line in proc.stdout:\n        print(line, end='')\n        match = re.search(r'https://[-a-zA-Z0-9.]+\\.trycloudflare\\.com', line)\n        if match and public_url is None:\n            public_url = match.group(0)\n\nthreading.Thread(target=read_logs, daemon=True).start()\n\nfor _ in range(30):\n    if public_url:\n        break\n    time.sleep(1)\n\nif not public_url:\n    raise RuntimeError('No se pudo obtener la URL pública todavía. Revisa los logs de cloudflared arriba.')\n\nprint('\nCOPIA ESTA URL EN EL .env DE MINA:')\nprint('VOICE_REMOTE_URL=' + public_url)\n